# Evaluate CAMeL-BERT on OpenITI Targeted Corpus

Test le modèle fine-tuné sur le corpus OpenITI ciblé dans Google Drive.

**Objectif:** Extraire des akhbars du corpus et évaluer la qualité
**Corpus:** /MyDrive/openiti_targeted (corpus spécifique)
**Modèle:** checkpoints/camelbert_akhbars_v2

## 1. Setup Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("[OK] Google Drive mounted!")

## 2. Setup repo and model

In [ ]:
import os
import subprocess
from pathlib import Path

# Paths
repo_path = '/content/Khabar-segmentation'
drive_repo_path = '/content/drive/MyDrive/Khabar-segmentation'
corpus_path = '/content/drive/MyDrive/openiti_targeted'
model_path = '/content/drive/MyDrive/Khabar-segmentation-results/checkpoints/camelbert_akhbars_v2'

# Copy repo if needed
if not os.path.exists(repo_path):
    print("[*] Copying repo from Drive...")
    subprocess.run(['cp', '-r', drive_repo_path, repo_path])
    print(f"[OK] Repo copied")
else:
    print(f"[OK] Repo already exists")

os.chdir(repo_path)

# Verify paths
print(f"\n[PATHS]")
print(f"  Repo: {repo_path}")
print(f"  Model: {model_path} {'✓' if os.path.exists(model_path) else '✗'}")
print(f"  Corpus: {corpus_path} {'✓' if os.path.exists(corpus_path) else '✗'}")

## 3. Install dependencies

In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pyarabic tqdm rapidfuzz

## 4. Load model and corpus

In [ ]:
import json
import re
from pathlib import Path
import torch
from transformers import pipeline
import numpy as np
from tqdm import tqdm

# Load model
print("[*] Loading CAMeL-BERT model...")
model_path = '/content/drive/MyDrive/Khabar-segmentation-results/checkpoints/camelbert_akhbars_v2'

nlp = pipeline(
    'token-classification',
    model=model_path,
    tokenizer='CAMeL-Lab/bert-base-arabic-camelbert-ca',
    device=0 if torch.cuda.is_available() else -1
)

print("[OK] Model loaded!")

# Find text files in corpus
corpus_path = Path('/content/drive/MyDrive/openiti_targeted')
text_files = list(corpus_path.rglob('*.txt')) + list(corpus_path.rglob('*.ara1'))
print(f"\n[*] Found {len(text_files)} text files in corpus")

if text_files:
    print(f"  Examples:")
    for f in text_files[:5]:
        print(f"    - {f.name}")

## 5. Clean OpenITI text

In [ ]:
def clean_openiti_text(text: str) -> str:
    """Clean OpenITI text: keep ONLY Arabic."""
    # Remove all # symbols
    text = text.replace('#', ' ')
    
    # Keep ONLY Arabic characters and spaces
    text = re.sub(
        r'[^\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF\s]',
        ' ',
        text
    )
    
    # Normalize spaces
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


def split_into_chunks(text: str, chunk_size: int = 1500, overlap: int = 300):
    """Split text into overlapping chunks."""
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(text):
            break
    return chunks


def extract_akhbars_from_text(text: str, confidence_threshold: float = 0.7):
    """Extract akhbars from text using token-level predictions."""
    # Clean text
    text = clean_openiti_text(text)
    
    if len(text) < 100:
        return []
    
    # Split into chunks
    chunks = split_into_chunks(text, chunk_size=1500, overlap=300)
    
    akhbars = []
    seen_texts = set()
    
    for chunk in chunks:
        if not chunk.strip():
            continue
        
        # Limit chunk size for pipeline
        try:
            predictions = nlp(chunk[:2000])
        except Exception as e:
            continue
        
        # Reconstruct akhbars from BIO tags
        current_akhbar = []
        current_scores = []
        
        for pred in predictions:
            token = pred.get('word', '')
            entity = pred.get('entity', 'O')
            score = pred.get('score', 0.0)
            
            # Handle AraBERT continuation tokens
            if token.startswith('##'):
                if current_akhbar:
                    current_akhbar[-1] += token[2:]
                continue
            
            if entity == 'B-KHABAR' and score >= confidence_threshold:
                # Save previous akhbar
                if current_akhbar:
                    akhbar_text = ' '.join(current_akhbar)
                    if akhbar_text not in seen_texts and len(akhbar_text) > 20:
                        avg_score = np.mean(current_scores)
                        akhbars.append({
                            'text': akhbar_text,
                            'confidence': float(avg_score),
                            'length': len(current_akhbar)
                        })
                        seen_texts.add(akhbar_text)
                
                # Start new akhbar
                current_akhbar = [token]
                current_scores = [score]
            
            elif entity == 'I-KHABAR' and score >= confidence_threshold and current_akhbar:
                current_akhbar.append(token)
                current_scores.append(score)
            
            else:
                # O or low confidence: end current akhbar
                if current_akhbar:
                    akhbar_text = ' '.join(current_akhbar)
                    if akhbar_text not in seen_texts and len(akhbar_text) > 20:
                        avg_score = np.mean(current_scores)
                        akhbars.append({
                            'text': akhbar_text,
                            'confidence': float(avg_score),
                            'length': len(current_akhbar)
                        })
                        seen_texts.add(akhbar_text)
                    current_akhbar = []
                    current_scores = []
        
        # Save last akhbar
        if current_akhbar:
            akhbar_text = ' '.join(current_akhbar)
            if akhbar_text not in seen_texts and len(akhbar_text) > 20:
                avg_score = np.mean(current_scores)
                akhbars.append({
                    'text': akhbar_text,
                    'confidence': float(avg_score),
                    'length': len(current_akhbar)
                })
                seen_texts.add(akhbar_text)
    
    return akhbars

print("[OK] Functions defined")

## 6. Evaluate on corpus

In [ ]:
import sys
sys.stdout.reconfigure(encoding='utf-8')

corpus_path = Path('/content/drive/MyDrive/openiti_targeted')
text_files = list(corpus_path.rglob('*.txt')) + list(corpus_path.rglob('*.ara1'))

print(f"[*] Evaluating on {len(text_files)} files...\n")

results = {
    'total_files': len(text_files),
    'total_akhbars': 0,
    'total_tokens': 0,
    'files_processed': 0,
    'avg_confidence': 0,
    'file_results': []
}

confidences = []

for file_path in tqdm(text_files, desc="Processing files"):
    try:
        # Load file
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        if len(text) < 100:
            continue
        
        # Extract akhbars
        akhbars = extract_akhbars_from_text(text, confidence_threshold=0.7)
        
        if akhbars:
            results['files_processed'] += 1
            results['total_akhbars'] += len(akhbars)
            results['total_tokens'] += sum(ak['length'] for ak in akhbars)
            
            for ak in akhbars:
                confidences.append(ak['confidence'])
            
            results['file_results'].append({
                'file': file_path.name,
                'akhbars_found': len(akhbars),
                'tokens': sum(ak['length'] for ak in akhbars),
                'avg_confidence': float(np.mean([ak['confidence'] for ak in akhbars]))
            })
    
    except Exception as e:
        continue

if confidences:
    results['avg_confidence'] = float(np.mean(confidences))

# Display results
print("\n" + "="*70)
print("[EVALUATION RESULTS]")
print("="*70)
print(f"Total files processed: {results['files_processed']}/{results['total_files']}")
print(f"Total akhbars extracted: {results['total_akhbars']}")
print(f"Total tokens: {results['total_tokens']}")
print(f"Average confidence: {results['avg_confidence']:.4f}")

print(f"\n[FILES WITH EXTRACTIONS]")
for fr in sorted(results['file_results'], key=lambda x: x['akhbars_found'], reverse=True)[:10]:
    print(f"  {fr['file']:40s} → {fr['akhbars_found']:3d} akhbars ({fr['avg_confidence']:.3f})")

print("="*70)

## 7. Save results

In [ ]:
import json
from datetime import datetime

# Save detailed results
output_path = Path('/content/drive/MyDrive/Khabar-segmentation-results/evaluation_openiti_targeted.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"[OK] Results saved to: {output_path}")
print(f"\n[SUMMARY]")
print(f"  Extraction quality: {'Good' if results['avg_confidence'] > 0.6 else 'Low'}")
print(f"  Files with results: {results['files_processed']}/{results['total_files']}")
print(f"  Akhbars found: {results['total_akhbars']}")